In [12]:
import numpy as np
import pandas as pd
import pickle as pkl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [3]:
# Loading Dataset
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Dropping the first 3 columns
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
# Encoding Categorical Columns

# 1. Encoding the "Gender" columns
encoder_gender = LabelEncoder()
df['Gender'] = encoder_gender.fit_transform(df['Gender'])
print(df['Gender'].value_counts())

Gender
1    5457
0    4543
Name: count, dtype: int64


In [15]:
ohe = OneHotEncoder()
geography_ohe = ohe.fit_transform(df[['Geography']])
geography_ohe

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [16]:
ohe.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [18]:
geography_ohe.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [20]:
geo_encoded_df = pd.DataFrame(geography_ohe.toarray(), columns=ohe.get_feature_names_out(["Geography"]))

In [21]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [22]:
# Add the above "geography" encoded values into the main dataframe
df = pd.concat([df.drop("Geography", axis=1), geo_encoded_df], axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [25]:
# Saving the encoders in pickle file
with open("label_encoder_gender.pkl", "wb") as file:
  pkl.dump(encoder_gender, file)
  
with open("one_hot_encoder_geography.pkl", "wb") as file:
  pkl.dump(geography_ohe, file)

In [26]:
# Divide the dataset into Independent and Dependent features
X = df.drop(columns=["Exited"])
y = df['Exited']

In [28]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling remaining features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [31]:
# X_train_scaled
# X_test_scaled

In [33]:
# Saving the "scaler" into .pkl file
with open("scaler.pkl", "wb") as file:
  pkl.dump(scaler, file)

In [34]:
# ANN Implementation
import keras
import datetime
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard

In [37]:
(X_train_scaled.shape[1], )

(12,)

In [38]:
# Building ANN Model
ann = Sequential([
  Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)), # First Hidden Layer
  Dense(32, activation="relu"), # Second Hidden Layer
  Dense(1, activation="sigmoid") # Output Layer
])

In [40]:
ann.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
# ANN Compilation - Method 1
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [42]:
# Other Method for Compiling a model - Method 2
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss = keras.losses.binary_crossentropy

ann.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])

In [60]:
# Setting up "TensorBoard" - To visualize the logs while training the model
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") 
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [48]:
# Setting up "Early Stopping"
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True)

In [49]:
# Train the model
history = ann.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs=100, callbacks=[tensorflow_callback, early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8577 - loss: 0.3398 - val_accuracy: 0.8590 - val_loss: 0.3463
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8599 - loss: 0.3374 - val_accuracy: 0.8615 - val_loss: 0.3432
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8621 - loss: 0.3336 - val_accuracy: 0.8610 - val_loss: 0.3465
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8615 - loss: 0.3321 - val_accuracy: 0.8640 - val_loss: 0.3409
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8641 - loss: 0.3293 - val_accuracy: 0.8580 - val_loss: 0.3440
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8652 - loss: 0.3280 - val_accuracy: 0.8605 - val_loss: 0.3419
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8669 - loss: 0.3261 - val_accuracy: 0.8605 - val_loss: 0.3395
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8654 - loss: 0.3236 - val_accu

In [50]:
ann.save("model.h5")

In [62]:
# Load tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [63]:
%reload_ext tensorboard

In [64]:
%tensorboard --logdir logs/fit

ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.